# NOVA Hybrid Continuous Learning — Multi-Asset / Multi-Timeframe

**Purpose:** reproducible research orchestration for Kaggle Free CPU. Each `(asset, timeframe)` is validated and modeled independently; calibrated out-of-fold (OOF) evidence is combined only after provenance, temporal, cost and risk gates. This notebook is not a broker client or execution engine.

**Hard safety invariants:** `RUN_TRAINING=False`, `PERSIST_ARTIFACTS=False`, `AUTO_PROMOTE=False`; no broker credentials, order APIs or live execution. Final OOS starts `2026-07-01T00:00:00Z` and is never loaded into training. Max orders remains an external EA safety limit of 5; this notebook cannot change risk policy.

**Important:** missing history is never synthesized. Unexplained gaps fail closed. Strategy-family predictions (Wave3 / SMC / XGBoost / LSTM / other approved model families) must be timestamp-aligned OOF outputs; in-sample predictions cannot enter Meta-Learning.

## 1. Configuration & execution mode
Keep all execution flags disabled until the dataset, CI and human-approval gates have independently passed.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, math, os, random
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED); np.random.seed(SEED)
DATA_ROOT = Path('/kaggle/input/nova-approved-market-data')
OUTPUT_ROOT = Path('/kaggle/working/nova_hybrid_training')
FINAL_OOS_START = pd.Timestamp('2026-07-01T00:00:00Z')
ASSETS = ('XAUUSDm', 'BTCUSD', 'BTCUSDT', 'EURUSD')
TIMEFRAMES = ('M5', 'M15', 'M30', 'H1', 'H4', 'D1')
REQUESTED_STREAMS = ()  # Explicit (asset, timeframe) pairs required before training; empty means blocked.
LABEL_HORIZON_BARS = 5
RUN_TRAINING = False
PERSIST_ARTIFACTS = False
AUTO_PROMOTE = False
MAX_CPU_WORKERS = 1
print({'data_root_exists': DATA_ROOT.exists(), 'run_training': RUN_TRAINING,
       'persist_artifacts': PERSIST_ARTIFACTS, 'auto_promote': AUTO_PROMOTE,
       'final_oos_start_utc': FINAL_OOS_START.isoformat()})

## 2. Data contract & provenance
Each asset/timeframe input must have a CSV and adjacent JSON manifest. The manifest must bind the exact CSV SHA-256, row count, asset, timeframe, UTC range, source and completeness status. A human approval file must bind the same dataset SHA and source commit.

In [ ]:
TIMEFRAME_MINUTES = {'M5':5, 'M15':15, 'M30':30, 'H1':60, 'H4':240, 'D1':1440}
REQUIRED_COLUMNS = {'timestamp_utc','asset','timeframe','open','high','low','close','volume','volume_type'}

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def validate_market_frame(frame, asset, timeframe, manifest):
    missing = REQUIRED_COLUMNS - set(frame.columns)
    if missing: raise ValueError(f'missing columns: {sorted(missing)}')
    if frame.empty: raise ValueError('empty dataset is not training eligible')
    data = frame.copy()
    if set(data['asset'].astype(str)) != {asset}: raise ValueError('asset identity mismatch')
    if set(data['timeframe'].astype(str)) != {timeframe}: raise ValueError('timeframe identity mismatch')

    declared_volume_type = manifest.get('volume_type')
    if not isinstance(declared_volume_type, str) or not declared_volume_type.strip():
        raise ValueError('manifest volume_type is required')
    if data['volume_type'].isna().any():
        raise ValueError('volume_type must be explicit and consistent')
    observed_volume_types = set(data['volume_type'].astype(str).str.strip())
    if observed_volume_types != {declared_volume_type.strip()}:
        raise ValueError('row volume_type does not match manifest volume_type')

    source = manifest.get('source')
    source_commit = manifest.get('source_commit_sha')
    if not isinstance(source, str) or not source.strip():
        raise ValueError('manifest source provenance is required')
    if (not isinstance(source_commit, str) or len(source_commit) != 40
        or any(character not in '0123456789abcdef' for character in source_commit.lower())):
        raise ValueError('manifest source_commit_sha must be a 40-character Git SHA')

    try:
        raw_timestamps = [pd.Timestamp(value) for value in data['timestamp_utc']]
    except Exception as exc:
        raise ValueError('invalid UTC timestamp') from exc
    if any(value.tzinfo is None or value.utcoffset() is None for value in raw_timestamps):
        raise ValueError('market timestamps must include an explicit timezone')
    data['timestamp_utc'] = pd.to_datetime(raw_timestamps, utc=True, errors='coerce')
    if data['timestamp_utc'].isna().any(): raise ValueError('invalid UTC timestamp')
    for col in ('open','high','low','close','volume'):
        data[col] = pd.to_numeric(data[col], errors='coerce')
    if data[list(('open','high','low','close','volume'))].isna().any().any():
        raise ValueError('missing or non-numeric OHLCV')
    if not np.isfinite(data[['open','high','low','close','volume']].to_numpy()).all():
        raise ValueError('non-finite OHLCV')
    if (data[['open','high','low','close']] <= 0).any().any() or (data['volume'] < 0).any():
        raise ValueError('invalid price or negative volume')
    if declared_volume_type.strip() == 'tick_volume' and (data['volume'] % 1 != 0).any():
        raise ValueError('MT5 tick_volume must be integer-valued')
    if ((data['high'] < data[['open','close','low']].max(axis=1)) |
        (data['low'] > data[['open','close','high']].min(axis=1))).any():
        raise ValueError('invalid OHLC bounds')
    if data['timestamp_utc'].duplicated().any(): raise ValueError('duplicate timestamps')
    if not data['timestamp_utc'].is_monotonic_increasing: raise ValueError('timestamps not increasing')

    def canonical_utc(value, field_name):
        try:
            timestamp = pd.Timestamp(value)
        except Exception as exc:
            raise ValueError(f'{field_name} must be an ISO-8601 UTC timestamp') from exc
        if timestamp.tzinfo is None or timestamp.utcoffset() is None:
            raise ValueError(f'{field_name} must include an explicit timezone')
        return timestamp.tz_convert('UTC').isoformat()

    expected_start = canonical_utc(manifest.get('start_utc'), 'start_utc')
    expected_end = canonical_utc(manifest.get('end_utc'), 'end_utc')
    if data['timestamp_utc'].iloc[0].isoformat() != expected_start:
        raise ValueError('dataset start timestamp does not match manifest')
    if data['timestamp_utc'].iloc[-1].isoformat() != expected_end:
        raise ValueError('dataset end timestamp does not match manifest')

    expected = pd.Timedelta(minutes=TIMEFRAME_MINUTES[timeframe])
    intervals = data['timestamp_utc'].diff().dropna()
    if intervals.lt(expected).any():
        raise ValueError(f'off-cadence bars: timestamps are closer than {timeframe} interval')
    gaps = data.loc[data['timestamp_utc'].diff().gt(expected), 'timestamp_utc'].index
    observed = []
    for gap_index in gaps:
        pos = data.index.get_loc(gap_index)
        observed.append({'from_utc': data.iloc[pos-1]['timestamp_utc'].isoformat(),
                         'to_utc': data.iloc[pos]['timestamp_utc'].isoformat()})

    approved_gaps = manifest.get('verified_gap_intervals', [])
    if not isinstance(approved_gaps, list):
        raise ValueError('verified_gap_intervals must be a list')
    normalized_gaps = []
    for gap in approved_gaps:
        if not isinstance(gap, dict) or not {'from_utc','to_utc'}.issubset(gap):
            raise ValueError('each verified gap must include from_utc and to_utc')
        normalized_gaps.append({
            'from_utc': canonical_utc(gap['from_utc'], 'verified_gap_intervals.from_utc'),
            'to_utc': canonical_utc(gap['to_utc'], 'verified_gap_intervals.to_utc'),
        })
    if observed != normalized_gaps:
        raise ValueError(f'unexplained or unreviewed gaps: {observed[:10]}')

    # Start a new segment after every verified session gap; no feature/label/sequence may cross it.
    data['segment_id'] = data['timestamp_utc'].diff().gt(expected).cumsum().astype('int64')
    if manifest.get('completeness_status') != 'PASS' or manifest.get('training_eligible') is not True:
        raise ValueError('manifest is not training eligible')
    row_count = manifest.get('row_count')
    if isinstance(row_count, bool) or not isinstance(row_count, int) or row_count != len(data):
        raise ValueError('manifest row count mismatch')
    if data['timestamp_utc'].max() >= FINAL_OOS_START:
        raise ValueError('training input contains sealed Final OOS timestamps')
    return data.reset_index(drop=True), observed
def load_approved_datasets(root=DATA_ROOT):
    datasets, audit = {}, []
    if not root.exists(): return datasets, [{'status':'BLOCKED','reason':'approved Kaggle input directory not mounted'}]
    for csv_path in sorted(root.rglob('*.csv')):
        manifest_path = csv_path.with_suffix('.manifest.json')
        if not manifest_path.exists():
            audit.append({'file':str(csv_path),'status':'BLOCKED','reason':'manifest missing'}); continue
        try:
            manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
            asset, timeframe = str(manifest['asset']), str(manifest['timeframe'])
            if asset not in ASSETS or timeframe not in TIMEFRAMES: raise ValueError('asset/timeframe not allowlisted')
            digest = sha256_file(csv_path)
            if digest != manifest.get('sha256'): raise ValueError('CSV SHA-256 mismatch')
            approval_path = manifest_path.with_name(manifest_path.stem + '.approval.json')
            if not approval_path.exists(): raise ValueError('human approval receipt missing')
            approval = json.loads(approval_path.read_text(encoding='utf-8'))
            if approval.get('approved') is not True or approval.get('dataset_sha256') != digest:
                raise ValueError('approval is absent or not bound to dataset SHA-256')
            if approval.get('source_commit_sha') != manifest.get('source_commit_sha'):
                raise ValueError('approval/source commit mismatch')
            raw = pd.read_csv(csv_path)
            clean, gaps = validate_market_frame(raw, asset, timeframe, manifest)
            key = (asset, timeframe)
            if key in datasets: raise ValueError('duplicate dataset for asset/timeframe')
            datasets[key] = {'frame':clean,'manifest':manifest,'sha256':digest,'path':str(csv_path)}
            audit.append({'asset':asset,'timeframe':timeframe,'rows':len(clean),'sha256':digest,'gaps':len(gaps),'status':'PASS'})
        except Exception as exc:
            audit.append({'file':str(csv_path),'status':'BLOCKED','reason':str(exc)})
    return datasets, audit

datasets, data_audit = load_approved_datasets()
pd.DataFrame(data_audit)

## 3. Causal feature engineering & labels
Features are computed independently within each asset/timeframe stream. No forward-looking features, synthetic bars or cross-market row mixing. Labels whose forward horizon reaches the sealed Final OOS boundary are discarded.

In [ ]:
def make_features(frame, horizon=LABEL_HORIZON_BARS):
    # Calculate rolling features and forward labels independently within each gap-bounded segment.
    parts = []
    for _, segment in frame.groupby('segment_id', sort=True):
        data = segment.copy().sort_values('timestamp_utc').reset_index(drop=True)
        close = data['close'].astype(float)
        data['return_1'] = close.pct_change(1)
        data['return_5'] = close.pct_change(5)
        data['ema_12'] = close.ewm(span=12, adjust=False).mean()
        data['ema_26'] = close.ewm(span=26, adjust=False).mean()
        delta = close.diff()
        gain = delta.clip(lower=0).ewm(alpha=1/14, adjust=False).mean()
        loss = (-delta.clip(upper=0)).ewm(alpha=1/14, adjust=False).mean()
        rs = gain / loss.replace(0, np.nan)
        data['rsi_14'] = (100 - 100/(1+rs)).fillna(50)
        data['ema_spread_pct'] = (data['ema_12'] - data['ema_26']) / close
        data['range_pct'] = (data['high'] - data['low']) / close
        data['volume_ratio_20'] = data['volume'] / data['volume'].rolling(20, min_periods=20).mean()
        future_close = close.shift(-horizon)
        data['forward_return'] = future_close / close - 1
        data['label'] = np.where(data['forward_return'] > 0, 1, np.where(data['forward_return'] < 0, 0, np.nan))
        data['label_end_utc'] = data['timestamp_utc'].shift(-horizon)
        data = data.replace([np.inf, -np.inf], np.nan)
        data = data.dropna(subset=FEATURE_COLUMNS+['forward_return','label_end_utc'])
        data = data.loc[data['label_end_utc'] < FINAL_OOS_START].copy()
        parts.append(data)
    if not parts:
        return pd.DataFrame(columns=list(frame.columns)+FEATURE_COLUMNS+['forward_return','label','label_end_utc'])
    return pd.concat(parts, ignore_index=True)


FEATURE_COLUMNS = ['return_1','return_5','rsi_14','ema_spread_pct','range_pct','volume_ratio_20']
feature_sets = {key: make_features(value['frame']) for key,value in datasets.items()}
feature_summary = [{'asset':k[0],'timeframe':k[1],'samples':len(v),'start':v.timestamp_utc.min().isoformat() if len(v) else None,
                    'end':v.timestamp_utc.max().isoformat() if len(v) else None} for k,v in feature_sets.items()]
pd.DataFrame(feature_summary)


## 4. Purged chronological walk-forward splits
Each market stream gets its own time-ordered folds. The label horizon is purged at the train/validation boundary. Random train/test splitting is prohibited.

In [ ]:
def walk_forward_folds(frame, folds=3, horizon=LABEL_HORIZON_BARS):
    if folds < 2 or horizon < 1 or len(frame) < (folds + 1) * 30:
        raise ValueError('insufficient samples or invalid walk-forward configuration')
    required = {'timestamp_utc', 'label_end_utc'}
    if required - set(frame.columns):
        raise ValueError('walk-forward frame requires timestamp_utc and label_end_utc')
    ordered = frame.sort_values('timestamp_utc').reset_index(drop=True)
    if ordered['timestamp_utc'].isna().any() or ordered['label_end_utc'].isna().any():
        raise ValueError('walk-forward timestamps and label boundaries must be present')
    if not ordered['timestamp_utc'].is_monotonic_increasing:
        raise ValueError('walk-forward timestamps must be ordered')
    block = len(ordered) // (folds + 1)
    results = []
    for fold in range(folds):
        split_index = block * (fold + 1)
        valid_start_index = min(split_index + horizon, len(ordered))
        valid_end_index = min(block * (fold + 2), len(ordered))
        valid = ordered.iloc[valid_start_index:valid_end_index].copy()
        if valid.empty:
            continue
        validation_start = valid['timestamp_utc'].iloc[0]
        # Purge by the actual label end timestamp, not row count alone: dropped/irregular
        # feature rows can otherwise let a training label overlap the validation period.
        train = ordered.loc[
            (ordered['timestamp_utc'] < validation_start) &
            (ordered['label_end_utc'] < validation_start)
        ].copy()
        if len(train) and len(valid):
            if not (train['label_end_utc'] < validation_start).all():
                raise AssertionError('purged walk-forward leakage detected')
            results.append((train, valid))
    return results

fold_summary = []
for key, frame in feature_sets.items():
    try:
        folds = walk_forward_folds(frame)
        fold_summary.append({'asset':key[0],'timeframe':key[1],'folds':len(folds),
                             'train_rows_last':len(folds[-1][0]),'validation_rows_last':len(folds[-1][1]),'status':'READY_FOR_RESEARCH'})
    except ValueError as exc:
        fold_summary.append({'asset':key[0],'timeframe':key[1],'folds':0,'status':'BLOCKED','reason':str(exc)})
pd.DataFrame(fold_summary)

## 5. Model families & hybrid evidence contract
Train candidates independently by market stream. XGBoost and LSTM are supervised candidates. Wave3 and SMC are strategy-family evidence adapters, not labels. Meta-Learning must consume timestamp-aligned OOF predictions with calibration metadata; correlated models are clustered so duplicate evidence cannot inflate consensus. Do not replace the existing Quant Engine's canonical Wave3/SMC/Meta Decision implementations with notebook approximations.

In [ ]:
def train_xgboost_candidate(train_frame, valid_frame, seed=SEED):
    from xgboost import XGBClassifier
    train = train_frame.dropna(subset=FEATURE_COLUMNS+['label'])
    valid = valid_frame.dropna(subset=FEATURE_COLUMNS+['label'])
    if len(train) < 100 or len(valid) < 30 or train.label.nunique() < 2:
        raise ValueError('insufficient balanced training/validation evidence')
    model = XGBClassifier(n_estimators=250,max_depth=3,learning_rate=.03,subsample=.8,
                          colsample_bytree=.8,reg_lambda=2,objective='binary:logistic',
                          eval_metric='logloss',random_state=seed,n_jobs=MAX_CPU_WORKERS,
                          tree_method='hist',early_stopping_rounds=20)
    model.fit(train[FEATURE_COLUMNS],train.label.astype(int),
              eval_set=[(valid[FEATURE_COLUMNS],valid.label.astype(int))],verbose=False)
    return model

def train_lstm_candidate(train_frame, valid_frame, seed=SEED, epochs=8):
    import torch
    from torch import nn
    torch.manual_seed(seed); torch.set_num_threads(MAX_CPU_WORKERS)
    train = train_frame.dropna(subset=FEATURE_COLUMNS+['label'])
    valid = valid_frame.dropna(subset=FEATURE_COLUMNS+['label'])
    if len(train) < 100 or len(valid) < 30: raise ValueError('insufficient LSTM samples')
    # Train-only normalization; validation is transformed with train statistics.
    mean = train[FEATURE_COLUMNS].mean().to_numpy(dtype=np.float32)
    scale = train[FEATURE_COLUMNS].std().replace(0,1).to_numpy(dtype=np.float32)
    def sequences(part):
        seq_x, seq_y = [], []
        for _, segment in part.groupby('segment_id', sort=True):
            x = ((segment[FEATURE_COLUMNS].to_numpy(dtype=np.float32)-mean)/scale)
            y = segment.label.to_numpy(dtype=np.float32)
            for i in range(31,len(x)):
                seq_x.append(x[i-31:i+1]); seq_y.append(y[i])
        if not seq_x: return torch.empty((0,32,len(FEATURE_COLUMNS))), torch.empty((0,))
        return torch.tensor(np.asarray(seq_x)), torch.tensor(np.asarray(seq_y))
    tx,ty = sequences(train); vx,vy = sequences(valid)
    if len(tx)==0 or len(vx)==0: raise ValueError('empty sequence windows')
    class SmallLSTM(nn.Module):
        def __init__(self):
            super().__init__(); self.lstm=nn.LSTM(len(FEATURE_COLUMNS),32,batch_first=True)
            self.head=nn.Sequential(nn.LayerNorm(32),nn.Linear(32,1))
        def forward(self,x):
            out,_=self.lstm(x); return self.head(out[:,-1]).squeeze(-1)
    model=SmallLSTM(); optimizer=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.0001)
    loss_fn=nn.BCEWithLogitsLoss(); best=float('inf'); best_state=None; stale=0
    for epoch in range(epochs):
        model.train()
        for start in range(0,len(tx),64):
            optimizer.zero_grad(); loss=loss_fn(model(tx[start:start+64]),ty[start:start+64]); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
        model.eval()
        with torch.no_grad(): score=float(loss_fn(model(vx),vy).item())
        if not math.isfinite(score): raise ValueError('non-finite validation loss')
        if score < best: best=score; best_state={k:v.detach().clone() for k,v in model.state_dict().items()}; stale=0
        else:
            stale+=1
            if stale>=3: break
    if best_state is None: raise ValueError('no valid LSTM checkpoint')
    model.load_state_dict(best_state)
    return {'model':model,'train_mean':mean,'train_scale':scale,'best_validation_loss':best}

training_results = []
if RUN_TRAINING:
    if AUTO_PROMOTE or PERSIST_ARTIFACTS: raise RuntimeError('automatic promotion and artifact persistence are disabled by policy')
    if not REQUESTED_STREAMS: raise RuntimeError('set explicit REQUESTED_STREAMS before training')
    if set(datasets) != set(REQUESTED_STREAMS):
        raise RuntimeError(f'dataset coverage must exactly match REQUESTED_STREAMS; got={sorted(datasets)}, expected={sorted(REQUESTED_STREAMS)}')
    for key in REQUESTED_STREAMS:
        frame = feature_sets[key]
        folds = walk_forward_folds(frame)
        for fold_id, (train_part, valid_part) in enumerate(folds, start=1):
            try:
                xgb_model = train_xgboost_candidate(train_part, valid_part)
                xgb_valid = valid_part.dropna(subset=FEATURE_COLUMNS+['label'])
                xgb_accuracy = float((xgb_model.predict(xgb_valid[FEATURE_COLUMNS]) == xgb_valid.label.astype(int)).mean())
                training_results.append({'asset':key[0],'timeframe':key[1],'fold':fold_id,'family':'xgboost',
                                         'status':'CANDIDATE_TRAINED','validation_accuracy':xgb_accuracy})
                lstm_result = train_lstm_candidate(train_part, valid_part)
                training_results.append({'asset':key[0],'timeframe':key[1],'fold':fold_id,'family':'lstm',
                                         'status':'CANDIDATE_TRAINED','validation_loss':lstm_result['best_validation_loss']})
            except Exception as exc:
                training_results.append({'asset':key[0],'timeframe':key[1],'fold':fold_id,'status':'BLOCKED','reason':str(exc)})
else:
    print('Training skipped: RUN_TRAINING=False. This is an intentional safety default.')
pd.DataFrame(training_results)

## 6. OOF Meta-Learning & continuous-learning gate
The notebook does not manufacture strategy signals. A separate, approved evaluation job must supply aligned OOF probabilities from each eligible family, including Wave3 and SMC evidence, with calibration and model-version metadata. The existing Quant Engine Meta Decision gate remains authoritative.

In [ ]:
def validate_oof_bundle(oof, required_families=('xgboost','lstm','wave3','smc')):
    required = {'timestamp_utc','asset','timeframe','family','probability',
                'fold_id','model_version','calibration_id','is_oof'}
    missing_columns = required - set(oof.columns)
    if missing_columns:
        raise ValueError(f'missing OOF fields: {sorted(missing_columns)}')
    if oof.empty:
        raise ValueError('OOF bundle is empty')
    if not required_families or len(set(required_families)) != len(required_families):
        raise ValueError('required_families must be a non-empty unique collection')

    data = oof.copy()
    # Never coerce naïve timestamps to UTC: a timezone-less vote cannot be aligned safely.
    try:
        parsed_times = [pd.Timestamp(value) for value in data['timestamp_utc']]
    except Exception as exc:
        raise ValueError('invalid OOF timestamp') from exc
    if any(value.tzinfo is None or value.utcoffset() is None for value in parsed_times):
        raise ValueError('OOF timestamps must include an explicit timezone')
    data['timestamp_utc'] = pd.to_datetime(parsed_times, utc=True)
    if (data['timestamp_utc'] >= FINAL_OOS_START).any():
        raise ValueError('sealed Final OOS predictions cannot enter training/meta-learning')

    for column in ('asset','timeframe','family','fold_id','model_version','calibration_id'):
        if data[column].isna().any() or data[column].astype(str).str.strip().eq('').any():
            raise ValueError(f'OOF {column} must be non-empty')
    data['family'] = data['family'].astype(str).str.strip()
    data['probability'] = pd.to_numeric(data['probability'], errors='coerce')
    if data['probability'].isna().any() or not np.isfinite(data['probability'].to_numpy(dtype=float)).all():
        raise ValueError('OOF probabilities must be finite numeric values')
    if not data['probability'].between(0,1).all():
        raise ValueError('probability outside [0,1]')
    is_oof = data['is_oof'].astype(str).str.strip().str.lower().isin({'true','1'})
    if not is_oof.all():
        raise ValueError('in-sample predictions are prohibited')
    vote_key = ['timestamp_utc','asset','timeframe','family']
    if data.duplicated(vote_key).any():
        raise ValueError('duplicate OOF vote')

    observed_families = set(data['family'].unique())
    missing_families = set(required_families) - observed_families
    if missing_families:
        raise ValueError(f'missing strategy/model families: {sorted(missing_families)}')

    group_key = ['timestamp_utc','asset','timeframe']
    coverage = data.groupby(group_key)['family'].agg(lambda values: set(values))
    incomplete = [
        (key, sorted(set(required_families) - families))
        for key, families in coverage.items()
        if not set(required_families).issubset(families)
    ]
    if incomplete:
        raise ValueError(f'incomplete per-timestamp OOF model-family coverage: {incomplete[:5]}')
    return True
CONTINUOUS_LEARNING_POLICY = {
    'trigger':'new verified pre-cutoff data or documented drift event; never consume sealed Final OOS for training',
    'automatic_retrain':False,
    'automatic_promotion':False,
    'required_gates':['dataset_sha256','source_provenance','gap_validation','purged_walk_forward',
                      'OOF_calibration','cost_aware_backtest','baseline_comparison','risk_review','human_approval'],
    'final_oos_start_utc':FINAL_OOS_START.isoformat(),
}
print(json.dumps(CONTINUOUS_LEARNING_POLICY,indent=2))

## 7. Evaluation & registry handoff
Trading metrics must be computed from event-level fills with spread, commission, swap and slippage. Classification accuracy alone is not evidence of trading edge. Candidate metadata is retained for review; this notebook does not write to the production registry or activate a model.

In [ ]:
def evaluate_trade_ledger(ledger):
    required={'net_pnl_after_costs','closed_at_utc','asset','timeframe','model_version'}
    if required-set(ledger.columns): raise ValueError('trade ledger missing cost-aware fields')
    pnl=pd.to_numeric(ledger['net_pnl_after_costs'],errors='coerce')
    if pnl.isna().any(): raise ValueError('invalid net PnL')
    gross_profit=float(pnl[pnl>0].sum()); gross_loss=float(-pnl[pnl<0].sum())
    return {'closed_trades':int(len(pnl)),'net_profit':float(pnl.sum()),
            'profit_factor':gross_profit/gross_loss if gross_loss>0 else None,
            'positive_net_profit':bool(pnl.sum()>0)}

candidate_registry = {
    'status':'RESEARCH_ONLY', 'promotable':False, 'auto_promote':False,
    'artifacts_persisted':False, 'live_trading_enabled':False,
    'max_orders_enforced_by':'external MQL5 EA Safety Gate (limit 5)',
    'datasets':{f'{a}/{t}':v['sha256'] for (a,t),v in datasets.items()},
    'training_results':training_results,
}
print(json.dumps(candidate_registry,indent=2,default=str))
if PERSIST_ARTIFACTS:
    raise RuntimeError('Artifact persistence is disabled until a separate reviewed registry handoff is approved.')
if not datasets:
    print('BLOCKED: no approved multi-asset/timeframe datasets mounted; no training or promotion occurred.')

## 8. Decision & next steps
Do not call this pipeline trained or production-ready until approved data is mounted, notebook execution is inspected, exact-head CI passes, and independent walk-forward / OOS / cost-aware risk evidence is recorded. Final OOS remains sealed; Live Trading remains OFF.